# YOLO26 Keypoint Estimation: Real-Time Pose Estimation with Ultralytics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

In this notebook, we implement **keypoint (pose) estimation** using the **YOLO26** pose model from Ultralytics on images and videos.

**YOLO26** introduces key improvements for pose estimation:
- **Residual Log-Likelihood Estimation (RLE)** for more accurate keypoint localization
- **End-to-end NMS-free inference**
- **MuSGD optimizer** for stable training
- Detects 17 anatomical landmarks (COCO format)

---

## 1. Installation & Setup

In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import HTML, display
from base64 import b64encode
from tqdm.auto import tqdm
import os
import shutil
import zipfile
import urllib.request

# cv2_imshow is Colab-only; fall back to a plain display for other platforms
try:
    from google.colab.patches import cv2_imshow
except ImportError:
    def cv2_imshow(img):
        from IPython.display import Image
        _, buf = cv2.imencode('.png', img)
        display(Image(data=buf.tobytes()))

print("All imports successful!")

---
## 2. Setup Workspace and Download Input Files

All processing runs in a **local workspace folder** (`./yolo26_workspace/`) with an `inputs/` and `outputs/` structure. The sample images and videos are packaged as a single zip in a public **GitHub release**, so they download directly into `inputs/` with no authentication required.

In [ ]:
# Workspace
WORK_DIR     = os.path.abspath("./yolo26_workspace")
INPUT_DIR    = os.path.join(WORK_DIR, "inputs")
OUTPUT_DIR   = os.path.join(WORK_DIR, "outputs")
VIDEO_OUTPUT = os.path.join(OUTPUT_DIR, "videos")
IMAGE_OUTPUT = os.path.join(OUTPUT_DIR, "images")

for d in [INPUT_DIR, VIDEO_OUTPUT, IMAGE_OUTPUT]:
    os.makedirs(d, exist_ok=True)

print(f"📁 Workspace: {WORK_DIR}")
print(f"   inputs  : {INPUT_DIR}")
print(f"   outputs : {OUTPUT_DIR}")

In [ ]:
# Public GitHub release zip — no login / no auth required
RELEASE_URL = "https://github.com/spmallick/learnopencv/releases/download/YOLO26_Keypoint_Estimation/YOLO26_Keypoint_Estimation.zip"
ZIP_PATH    = os.path.join(WORK_DIR, "YOLO26_Keypoint_Estimation.zip")

MEDIA_EXT = ('.mp4', '.mov', '.avi', '.mkv',
             '.jpg', '.jpeg', '.png', '.bmp', '.webp')

existing = [f for f in os.listdir(INPUT_DIR)
            if os.path.isfile(os.path.join(INPUT_DIR, f))
            and f.lower().endswith(MEDIA_EXT)]

if existing:
    print(f"✅ Using {len(existing)} media file(s) already in {INPUT_DIR}")
else:
    # --- Download ---
    def _report(block_num, block_size, total_size):
        downloaded = block_num * block_size
        pct = min(downloaded * 100 / total_size, 100) if total_size > 0 else 0
        print(f"\r⬇️  Downloading release zip... {pct:5.1f}%  ({downloaded / (1024*1024):6.2f} MB)",
              end='', flush=True)

    print(f"⬇️  Downloading from: {RELEASE_URL}")
    urllib.request.urlretrieve(RELEASE_URL, ZIP_PATH, reporthook=_report)
    print()

    # --- Extract to a temp sub-folder so we can safely flatten afterwards ---
    TEMP_EXTRACT = os.path.join(WORK_DIR, "_zip_contents")
    os.makedirs(TEMP_EXTRACT, exist_ok=True)

    print("📦 Extracting zip...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(TEMP_EXTRACT)

    # --- Walk the extracted tree and move every media file into INPUT_DIR ---
    moved = 0
    for root, dirs, files in os.walk(TEMP_EXTRACT):
        for fname in files:
            if fname.lower().endswith(MEDIA_EXT):
                src = os.path.join(root, fname)
                dst = os.path.join(INPUT_DIR, fname)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
                    moved += 1

    # --- Clean up ---
    shutil.rmtree(TEMP_EXTRACT, ignore_errors=True)
    os.remove(ZIP_PATH)
    print(f"✅ Moved {moved} media file(s) into {INPUT_DIR}")

# List final inputs
files = sorted([f for f in os.listdir(INPUT_DIR)
                if os.path.isfile(os.path.join(INPUT_DIR, f))])
print(f"\n📦 {len(files)} file(s) in workspace:")
for f in files:
    size_mb = os.path.getsize(os.path.join(INPUT_DIR, f)) / (1024 * 1024)
    print(f"   - {f}  ({size_mb:.2f} MB)")

---
## 3. Choose & Load the YOLO26 Pose Model

YOLO26 comes in **5 variants** ranging from a tiny 2.9M-parameter nano model to a 57.6M-parameter extra-large one. Pick the variant that matches your **speed vs accuracy** needs:

| Model | Params (M) | FLOPs (B) | mAP₅₀₋₉₅ (Pose) | T4 GPU Speed (ms) | Best For |
|-------|-----------|-----------|----------------|-------------------|----------|
| **yolo26n-pose** | 2.9 | 7.5 | 57.2 | **1.8** ⚡ | Real-time on edge devices, webcams, mobile |
| **yolo26s-pose** | 10.4 | 23.9 | 63.0 | 2.7 | Balanced speed + accuracy on consumer GPU |
| **yolo26m-pose** | 21.5 | 73.1 | 68.8 | 5.0 | Higher accuracy with reasonable speed |
| **yolo26l-pose** | 25.9 | 91.3 | 70.4 | 6.5 | High accuracy for analysis pipelines |
| **yolo26x-pose** | 57.6 | 201.7 | **71.6** 🎯 | 12.2 | Maximum accuracy, offline batch processing |

*Benchmarks on COCO val2017 at 640×640. Speed measured on NVIDIA T4 GPU with TensorRT10.*

### How to choose:
- **Live demo / mobile / webcam?** → `yolo26n-pose` (fastest)
- **Blog tutorials / Colab T4?** → `yolo26n-pose` or `yolo26s-pose` (good balance)
- **Production pipeline?** → `yolo26m-pose` or `yolo26l-pose`
- **Research / batch analysis?** → `yolo26x-pose` (best accuracy)

For this tutorial, we'll use `yolo26n-pose.pt` since it's fast on Colab's free T4 GPU. Change the model name below to try a different variant.

In [ ]:
# Choose your model variant:
#   "yolo26n-pose.pt"   ← Nano   (fastest, smallest)
#   "yolo26s-pose.pt"   ← Small
#   "yolo26m-pose.pt"   ← Medium
#   "yolo26l-pose.pt"   ← Large
#   "yolo26x-pose.pt"   ← XLarge (most accurate, slowest)

MODEL_NAME = "yolo26n-pose.pt"

model = YOLO(MODEL_NAME)
print(f"✅ Loaded {MODEL_NAME}")

---
## 4. Helper Functions

In [ ]:
def display_video_inline(video_path, width=720, crf=18, preset='medium'):
    """
    Display a video inline (H.264 compressed for fast loading).
    crf: 18 = visually lossless, 22 = high quality (default), 28 = medium, 35 = low.
    """
    compressed_path = '/tmp/display_video.mp4'
    cmd = (
        f"ffmpeg -y -i '{video_path}' "
        f"-vcodec libx264 -crf {crf} -preset {preset} "
        f"-pix_fmt yuv420p "
        f"-vf 'scale={width}:-2' -an "
        f"-movflags +faststart "
        f"-f mp4 {compressed_path} -loglevel quiet"
    )
    os.system(cmd)
    with open(compressed_path, 'rb') as f:
        video_data = f.read()
    video_b64 = b64encode(video_data).decode()
    size_mb = len(video_data) / (1024 * 1024)
    print(f"🎬 {os.path.basename(video_path)} — {size_mb:.2f} MB (crf={crf}, width={width})")
    html = f'''
    <video width="{width}" controls autoplay muted loop>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''
    return HTML(html)


def display_image_inline(image, width=720, title=None):
    h, w = image.shape[:2]
    if w > width:
        scale = width / w
        image = cv2.resize(image, (width, int(h * scale)))
    if title:
        print(f"🖼️ {title}")
    cv2_imshow(image)


def run_keypoint_on_video(video_path, output_path, conf=0.5, plot_kwargs=None):
    """Run YOLO26 keypoint estimation on a video frame by frame with a progress bar."""
    if plot_kwargs is None:
        plot_kwargs = {'boxes': False, 'labels': False, 'conf': False}
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Could not open video: {video_path}")
        return
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"  Resolution: {width}x{height} @ {fps}fps | {total} frames")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    pbar = tqdm(total=total, desc=f"  {os.path.basename(video_path)}",
                unit="frame", dynamic_ncols=True, leave=True)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        results = model(frame, conf=conf, verbose=False)
        annotated = results[0].plot(**plot_kwargs)
        out.write(annotated)
        pbar.update(1)
    pbar.close()
    cap.release()
    out.release()
    print(f"  ✅ Saved: {os.path.basename(output_path)}")


print("✅ Helper functions ready")

---
## 5. Keypoint Estimation on Images

Let's run YOLO26 pose estimation on a variety of activities — yoga, martial arts, gym workouts, walking, gymnastics, group play, and sports action — to see how the model performs across different scenarios.

### 5.1 Karate

Karate is a martial art focused on quick, controlled striking movements such as kicks, punches, and blocks. Practitioners often hold extreme stances with one leg fully raised — a great test for pose estimation under dynamic body configurations.

In [ ]:
image_name = "karate.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

results = model(input_path, conf=0.5, verbose=False)
annotated = results[0].plot()  # default — boxes + labels + confidence

output_path = os.path.join(IMAGE_OUTPUT, f"output_with_box_{image_name}")
cv2.imwrite(output_path, annotated)
display_image_inline(annotated, width=700, title=f"YOLO26 Pose on {image_name}")
print(f"✅ Saved to: {output_path}")

Beyond visualization, you can extract raw keypoint coordinates and confidence scores for downstream tasks like joint angle calculation, action classification, or fitness rep counting.

In [ ]:
KEYPOINT_NAMES = [
    "Nose", "Left Eye", "Right Eye", "Left Ear", "Right Ear",
    "Left Shoulder", "Right Shoulder", "Left Elbow", "Right Elbow",
    "Left Wrist", "Right Wrist", "Left Hip", "Right Hip",
    "Left Knee", "Right Knee", "Left Ankle", "Right Ankle"
]

image_name = "karate.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

results = model(input_path, conf=0.5, verbose=False)
for result in results:
    if result.keypoints is not None:
        for person_idx, kps in enumerate(result.keypoints.xy):
            print(f"\n--- Person {person_idx + 1} ---")
            for i, (x, y) in enumerate(kps):
                if x > 0 and y > 0:
                    conf_score = result.keypoints.conf[person_idx][i].item()
                    print(f"  {KEYPOINT_NAMES[i]:20s}: ({x:.1f}, {y:.1f})  conf={conf_score:.3f}")

### 5.2 Yoga

Yoga is a mind-body practice involving controlled breathing and complex postures that demand exceptional flexibility, balance, and body awareness. Unusual joint configurations — inverted poses, twists, and balancing on a single limb — make it a challenging test for any pose estimator.

In [ ]:
image_name = "Yoga.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

original = cv2.imread(input_path)
results = model(original, conf=0.5, verbose=False)
annotated = results[0].plot(boxes=False, labels=False, conf=False)

h, w = original.shape[:2]
display_h = 500
scale = display_h / h
display_w = int(w * scale)

original_resized  = cv2.resize(original,  (display_w, display_h))
annotated_resized = cv2.resize(annotated, (display_w, display_h))

cv2.putText(original_resized,  "Original Image",         (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)
cv2.putText(annotated_resized, "YOLO26 Pose Estimation", (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)

comparison = np.hstack([original_resized, annotated_resized])
output_path = os.path.join(IMAGE_OUTPUT, f"comparison_{image_name}")
cv2.imwrite(output_path, comparison)
display_image_inline(comparison, width=1200, title=image_name)
print(f"✅ Saved to: {output_path}")

### 5.3 Acro Yoga

Acro Yoga combines yoga and acrobatics with two or more partners performing balance-based poses. Multiple persons in close contact, overlapping limbs, and unusual orientations make it one of the toughest scenarios for pose estimation.

In [ ]:
image_name = "Acro Yoga.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

results = model(input_path, conf=0.5, verbose=False)
annotated = results[0].plot(
    boxes=False, labels=False, conf=False,
    line_width=3, kpt_radius=6,
)

output_path = os.path.join(IMAGE_OUTPUT, f"output_styled_{image_name}")
cv2.imwrite(output_path, annotated)
display_image_inline(annotated, width=700, title=f"YOLO26 Pose on {image_name}")
print(f"✅ Saved to: {output_path}")

### 5.4 Gym Workout

Gym workouts involve repetitive movements like squats, presses, and lifts. Pose estimation here has practical value — it can be used to check exercise form, count reps, or detect imbalance between left and right sides.

In [ ]:
image_name = "Gym.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

original = cv2.imread(input_path)
results = model(original, conf=0.5, verbose=False)
annotated = results[0].plot(boxes=False, labels=False, conf=False)

h, w = original.shape[:2]
display_h = 500
scale = display_h / h
display_w = int(w * scale)

original_resized  = cv2.resize(original,  (display_w, display_h))
annotated_resized = cv2.resize(annotated, (display_w, display_h))

cv2.putText(original_resized,  "Original Image",         (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)
cv2.putText(annotated_resized, "YOLO26 Pose Estimation", (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)

comparison = np.hstack([original_resized, annotated_resized])
output_path = os.path.join(IMAGE_OUTPUT, f"comparison_{image_name}")
cv2.imwrite(output_path, comparison)
display_image_inline(comparison, width=1200, title=image_name)
print(f"✅ Saved to: {output_path}")

### 5.5 Walking

Walking is the most basic human motion — a continuous gait cycle of alternating leg swings and arm counter-swings. While it looks simple, it serves as a baseline test: if pose estimation cannot reliably track walking, it cannot track anything more complex.

In [ ]:
image_name = "Walk.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

results = model(input_path, conf=0.5, verbose=False)
annotated = results[0].plot(boxes=False, labels=False, conf=False)

output_path = os.path.join(IMAGE_OUTPUT, f"output_{image_name}")
cv2.imwrite(output_path, annotated)
display_image_inline(annotated, width=700, title=f"YOLO26 Pose on {image_name}")
print(f"✅ Saved to: {output_path}")

### 5.6 Gymnastics

Gymnastics features some of the most extreme human body configurations — inverted bodies, mid-air rotations, and full-body extensions on apparatus like the rings or bars. The body orientation can be entirely flipped or twisted, often confusing keypoint estimators trained on standard upright poses.

In [ ]:
image_name = "Gymnastics.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

original = cv2.imread(input_path)
results = model(original, conf=0.5, verbose=False)
annotated = results[0].plot(boxes=False, labels=False, conf=False, line_width=3, kpt_radius=6)

h, w = original.shape[:2]
display_h = 500
scale = display_h / h
display_w = int(w * scale)

original_resized  = cv2.resize(original,  (display_w, display_h))
annotated_resized = cv2.resize(annotated, (display_w, display_h))

cv2.putText(original_resized,  "Original Image",         (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)
cv2.putText(annotated_resized, "YOLO26 Pose Estimation", (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)

comparison = np.hstack([original_resized, annotated_resized])
output_path = os.path.join(IMAGE_OUTPUT, f"comparison_{image_name}")
cv2.imwrite(output_path, comparison)
display_image_inline(comparison, width=1200, title=image_name)
print(f"✅ Saved to: {output_path}")

### 5.7 Group Play

Group play scenarios often involve multiple people interacting closely together — children playing, friends in a casual moment, or team activities. Detecting every person individually in such crowded scenes is a strong test of multi-person pose estimation.

In [ ]:
image_name = "Play.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

results = model(input_path, conf=0.5, verbose=False)
annotated = results[0].plot(boxes=True, labels=False, conf=False)

if results[0].keypoints is not None:
    n_people = len(results[0].keypoints.xy)
    print(f"👥 Detected {n_people} person(s) in this frame")

output_path = os.path.join(IMAGE_OUTPUT, f"output_{image_name}")
cv2.imwrite(output_path, annotated)
display_image_inline(annotated, width=700, title=f"YOLO26 Pose on {image_name}")
print(f"✅ Saved to: {output_path}")

### 5.8 Sports Action

Sports action shots capture fast-paced moments mid-action — a player throwing, jumping, or running. These shots often have motion blur, partial occlusions from equipment, and players in non-standard positions. A great real-world test of how robust pose estimation is for sports analytics applications.

In [ ]:
image_name = "Game.jpg"
input_path = os.path.join(INPUT_DIR, image_name)

results = model(input_path, conf=0.5, verbose=False)
annotated = results[0].plot(boxes=False, labels=False, conf=False)

output_path = os.path.join(IMAGE_OUTPUT, f"output_{image_name}")
cv2.imwrite(output_path, annotated)
display_image_inline(annotated, width=700, title=f"YOLO26 Pose on {image_name}")
print(f"✅ Saved to: {output_path}")

---
## 6. Keypoint Estimation on Videos

Static images are great, but pose estimation truly shines on video. Let's see how YOLO26 tracks the human skeleton through fast-paced movements like boxing, yoga, parkour, dance, and jumps.

### 6.1 Boxing

Boxing is a fast-paced combat sport with rapid arm movements, frequent occlusions when fists cross the body, and dynamic weight shifts. Tracking the punching arm consistently is a strong test of pose estimation under motion.

In [ ]:
video_name = "Boxing.mp4"
input_path  = os.path.join(INPUT_DIR, video_name)
output_path = os.path.join(VIDEO_OUTPUT, f"output_{video_name}")

print(f"--- Processing {video_name} ---")
run_keypoint_on_video(input_path, output_path, conf=0.5,
                      plot_kwargs={'boxes': False, 'labels': False, 'conf': False})

In [ ]:
display_video_inline(output_path, width=560)

### 6.2 Yoga

Yoga in motion involves smooth transitions between asanas (poses). Each pose tests a different aspect of flexibility, balance, and stillness — all of which the model needs to track frame by frame.

In [ ]:
video_name = "Yoga.mp4"
input_path  = os.path.join(INPUT_DIR, video_name)
output_path = os.path.join(VIDEO_OUTPUT, f"output_with_box_{video_name}")

print(f"--- Processing {video_name} ---")
run_keypoint_on_video(input_path, output_path, conf=0.5,
                      plot_kwargs={'boxes': True, 'labels': False, 'conf': False})

In [ ]:
display_video_inline(output_path)

### 6.3 Parkour

Parkour involves explosive jumps, vaults, and flips through urban environments. The model has to track extreme body configurations — fully extended limbs, mid-air rotations, tucked positions — making this an extreme stress test for any pose estimator.

In [ ]:
video_name = "Parkour.mp4"
input_path  = os.path.join(INPUT_DIR, video_name)
output_path = os.path.join(VIDEO_OUTPUT, f"output_styled_{video_name}")

print(f"--- Processing {video_name} ---")
run_keypoint_on_video(input_path, output_path, conf=0.5,
                      plot_kwargs={'boxes': False, 'labels': False, 'conf': False,
                                   'line_width': 4, 'kpt_radius': 7})

In [ ]:
display_video_inline(output_path)

### 6.4 Dance

Dance combines rhythm, style, and full-body coordination. Limbs swing in wide arcs and the body changes orientation rapidly — making it a natural fit for showing off pose estimation on creative human movement.

In [ ]:
video_name = "Dance.mp4"
input_path  = os.path.join(INPUT_DIR, video_name)
output_path = os.path.join(VIDEO_OUTPUT, f"output_{video_name}")

cap = cv2.VideoCapture(input_path)
fps    = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"  Resolution: {width}x{height} @ {fps}fps | {total} frames")

# Output width is 2x original (two frames side by side)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width * 2, height))

pbar = tqdm(total=total, desc=f"  {video_name}",
            unit="frame", dynamic_ncols=True, leave=True)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, conf=0.5, verbose=False)
    annotated = results[0].plot(boxes=False, labels=False, conf=False)

    original_labeled = frame.copy()
    cv2.putText(original_labeled, "Original Video", (15, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3, cv2.LINE_AA)
    cv2.putText(annotated, "YOLO26 Pose Estimation", (15, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3, cv2.LINE_AA)

    combined = np.hstack([original_labeled, annotated])
    out.write(combined)
    pbar.update(1)

pbar.close()
cap.release()
out.release()
print(f"  ✅ Saved: {os.path.basename(output_path)}")

In [ ]:
display_video_inline(output_path, width=960)

### 6.5 Jump

Jumping captures the human body at moments of full extension — arms above the head, legs bent or straight, the body suspended in air with no ground contact. The body's center of gravity, posture, and limb positions all change rapidly through standing, takeoff, peak, and landing.

In [ ]:
video_name = "Jump.mp4"
input_path  = os.path.join(INPUT_DIR, video_name)
output_path = os.path.join(VIDEO_OUTPUT, f"output_{video_name}")

print(f"--- Processing {video_name} ---")
run_keypoint_on_video(input_path, output_path, conf=0.5,
                      plot_kwargs={'boxes': False, 'labels': False, 'conf': False})

In [ ]:
display_video_inline(output_path)

### 6.6 Choreography Progression

A single frame doesn't tell the full story of a dance routine. By extracting four evenly-spaced keyframes across the video and arranging them in a 2×2 grid, we can see how the pose evolves over time — useful for choreography analysis or showing the temporal flow of a routine in a single image.

In [ ]:
video_name = "Dance 2.mp4"
input_path = os.path.join(INPUT_DIR, video_name)

cap = cv2.VideoCapture(input_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
positions = [0.10, 0.35, 0.60, 0.85]
frames_annotated = []

for pos in positions:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(total_frames * pos))
    ret, frame = cap.read()
    if not ret:
        continue
    results = model(frame, conf=0.5, verbose=False)
    annotated = results[0].plot(boxes=False, labels=False, conf=False, line_width=3, kpt_radius=5)
    cv2.putText(annotated, f"t = {int(pos * 100)}%", (15, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
    frames_annotated.append(annotated)

cap.release()

if len(frames_annotated) == 4:
    h = 350
    resized = []
    for f in frames_annotated:
        fh, fw = f.shape[:2]
        scale = h / fh
        resized.append(cv2.resize(f, (int(fw * scale), h)))
    min_w = min(f.shape[1] for f in resized)
    resized = [f[:, :min_w] for f in resized]
    top    = np.hstack([resized[0], resized[1]])
    bottom = np.hstack([resized[2], resized[3]])
    grid   = np.vstack([top, bottom])
    grid_path = os.path.join(VIDEO_OUTPUT, f"keyframes_{video_name.replace('.mp4', '.jpg')}")
    cv2.imwrite(grid_path, grid)
    display_image_inline(grid, width=1200, title=f"Pose progression in {video_name}")
    print(f"✅ Saved keyframe grid to: {grid_path}")
else:
    print(f"❌ Could only extract {len(frames_annotated)} frames")

---
## 7. COCO Keypoint Reference

| Index | Keypoint | Index | Keypoint |
|-------|----------|-------|----------|
| 0 | Nose | 9 | Left Wrist |
| 1 | Left Eye | 10 | Right Wrist |
| 2 | Right Eye | 11 | Left Hip |
| 3 | Left Ear | 12 | Right Hip |
| 4 | Right Ear | 13 | Left Knee |
| 5 | Left Shoulder | 14 | Right Knee |
| 6 | Right Shoulder | 15 | Left Ankle |
| 7 | Left Elbow | 16 | Right Ankle |
| 8 | Right Elbow | | |

---
## Summary

We implemented YOLO26 keypoint estimation on **8 images** and **6 videos** with multiple visualization styles.

### What makes YOLO26 special?
- **RLE (Residual Log-Likelihood Estimation)** for accurate keypoint localization
- **End-to-end NMS-free inference** for faster deployment
- **Up to 43% faster CPU inference** vs predecessors
- **Non-human keypoint support** — works beyond human poses

### References
- [Ultralytics YOLO26 Documentation](https://docs.ultralytics.com/models/yolo26/)
- [Pose Estimation Docs](https://docs.ultralytics.com/tasks/pose/)
- [YOLO26: An Analysis of NMS-Free End-to-End Framework for Real-Time Object Detection](https://arxiv.org/abs/2601.12882)
- [LearnOpenCV Blog](https://learnopencv.com/)